# 📡 Cliente de Sensores — Enviar y Consultar Datos

Este notebook simula un dispositivo (como el ESP32 de la diapositiva *'El Viaje del Mensaje'*) que envía lecturas de un sensor al servidor, y luego las consulta desde la base de datos.

Aplica directamente a los proyectos de referencia: *Diagnóstico Asistido por IA* (fonocardiograma) y *Fotopletismografía por Cámara* (PPG).

In [ ]:
IP_SERVIDOR = "TU-IP-PUBLICA"
PUERTO = 8000
BASE_URL = f"http://{IP_SERVIDOR}:{PUERTO}"

import requests
print("Conectando a:", BASE_URL)

## 1. Enviar una lectura simulada — POST `/sensor/lectura`

In [ ]:
import random

lectura = {
    "alumno": "tu-nombre-aqui",
    "sensor": "ppg",       # prueba también "fonocardiograma", "temperatura", etc.
    "valor": round(random.uniform(60, 100), 2),  # ej: pulsaciones por minuto
}

respuesta = requests.post(f"{BASE_URL}/sensor/lectura", json=lectura)
print(respuesta.status_code)
respuesta.json()

## 2. Enviar varias lecturas seguidas

Simulamos una pequeña serie de tiempo, como si el sensor estuviera midiendo cada segundo.

In [ ]:
import time

for i in range(10):
    lectura = {
        "alumno": "tu-nombre-aqui",
        "sensor": "ppg",
        "valor": round(70 + 10 * random.random(), 2),
    }
    r = requests.post(f"{BASE_URL}/sensor/lectura", json=lectura)
    print(f"Lectura {i+1}/10 enviada -> status {r.status_code}")
    time.sleep(0.3)

## 3. Consultar el historial guardado — GET `/sensor/historial`

In [ ]:
respuesta = requests.get(
    f"{BASE_URL}/sensor/historial",
    params={"alumno": "tu-nombre-aqui", "limite": 20},
)
datos = respuesta.json()
print("Total de lecturas:", datos["total"])
datos["lecturas"]

## 4. Graficar los valores recibidos

In [ ]:
import matplotlib.pyplot as plt

valores = [l["valor"] for l in reversed(datos["lecturas"])]

plt.figure(figsize=(8, 4))
plt.plot(valores, marker="o")
plt.title("Lecturas recibidas desde el servidor en la nube")
plt.xlabel("Lectura #")
plt.ylabel("Valor")
plt.grid(True, alpha=0.3)
plt.show()

## 🧪 Reto para el alumno

1. Cambia `sensor` a otro nombre y filtra el historial solo por ese sensor.
2. Agrega tu propio `alumno` (tu nombre) y compara tu historial con el de un compañero que use el mismo servidor.
3. ¿Qué pasaría si dos alumnos envían datos al mismo tiempo? Investiga el concepto de *condiciones de carrera* (race conditions).